In [0]:
# Python utilities used for JSON handling, file paths and batch timestamps.
import json
from datetime import datetime, timezone
from pathlib import Path

# Spark functions and data types used to create the Bronze dataframe.
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType,
)


# Stable project identifier used in lineage and audit fields.
PROJECT_NAME = "trichy_east_election_pipeline"


# Unity Catalog locations for the Bronze table.
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"


# Root Databricks Volume containing year-specific source folders.
BASE_VOLUME_PATH = "/Volumes/workspace/elections/trichy_east"


# Stable constituency metadata added to every Bronze record.
CONSTITUENCY_NO = "141"
CONSTITUENCY_NAME = "Tiruchirappalli East"


# This notebook processes only official candidate-total sources.
SOURCE_TYPE = "candidate_totals"


# Unique run identifier used to trace every row to this execution.
BATCH_ID = datetime.now(timezone.utc).strftime(
    "candidate_totals_%Y%m%dT%H%M%SZ"
)


# Target raw Delta table.
BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}."
    "trichy_east_candidate_totals_raw"
)


# Git-managed source configuration.
MANIFEST_PATH = Path("config/source_manifest.json")


print("Project:", PROJECT_NAME)
print("Source type:", SOURCE_TYPE)
print("Batch ID:", BATCH_ID)
print("Target table:", BRONZE_TABLE)
print("Manifest:", MANIFEST_PATH)

In [0]:
# Stop early if the Git configuration has not been synced.
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Source manifest was not found at {MANIFEST_PATH}. "
        "Pull the latest Git changes in Databricks."
    )


# Read the configuration maintained in Git.
with MANIFEST_PATH.open("r") as manifest_file:
    source_manifest = json.load(manifest_file)


# Confirm that the notebook and manifest use the same Volume.
assert (
    source_manifest["databricks_base_volume"]
    == BASE_VOLUME_PATH
), "The notebook and manifest Volume paths do not match."


# Select only candidate-total source files.
candidate_total_source_rows = []

for source in source_manifest["sources"]:
    if source["source_type"] != SOURCE_TYPE:
        continue

    source_file_path = (
        f"{BASE_VOLUME_PATH.rstrip('/')}/"
        f"{source['relative_path'].lstrip('/')}"
    )

    candidate_total_source_rows.append(
        {
            "project_name": PROJECT_NAME,
            "constituency_no": CONSTITUENCY_NO,
            "constituency_name": CONSTITUENCY_NAME,
            "election_year": int(source["election_year"]),
            "source_type": source["source_type"],
            "source_format": source["format"],
            "source_file_name": source["file_name"],
            "source_file_path": source_file_path,
            "configured_extraction_strategy": source[
                "extraction_strategy"
            ],
            "batch_id": BATCH_ID,
        }
    )


# Convert the selected configuration into a Spark dataframe.
candidate_total_source_df = spark.createDataFrame(
    candidate_total_source_rows
)


display(
    candidate_total_source_df.orderBy(
        "election_year"
    )
)


print(
    "Configured candidate-total files:",
    candidate_total_source_df.count()
)

In [0]:
# Python utility used to inspect files mounted under /Volumes.
import os


candidate_total_file_check_rows = []


for source in candidate_total_source_rows:
    source_file_path = source["source_file_path"]

    try:
        file_exists = os.path.isfile(source_file_path)

        if file_exists:
            file_size_bytes = int(
                os.path.getsize(source_file_path)
            )
            file_status = "AVAILABLE"
            validation_error = None
        else:
            file_size_bytes = None
            file_status = "MISSING"
            validation_error = (
                "Configured candidate-total file was not found."
            )

    except Exception as error:
        file_exists = False
        file_size_bytes = None
        file_status = "ERROR"
        validation_error = str(error)

    candidate_total_file_check_rows.append(
        {
            **source,
            "file_exists": file_exists,
            "file_size_bytes": file_size_bytes,
            "file_status": file_status,
            "validation_error": validation_error,
        }
    )


# Explicit types prevent Spark from guessing the type of all-null errors.
candidate_total_file_check_schema = StructType([
    StructField("project_name", StringType(), False),
    StructField("constituency_no", StringType(), False),
    StructField("constituency_name", StringType(), False),
    StructField("election_year", IntegerType(), False),
    StructField("source_type", StringType(), False),
    StructField("source_format", StringType(), False),
    StructField("source_file_name", StringType(), False),
    StructField("source_file_path", StringType(), False),
    StructField(
        "configured_extraction_strategy",
        StringType(),
        False
    ),
    StructField("batch_id", StringType(), False),
    StructField("file_exists", StringType(), False),
    StructField("file_size_bytes", StringType(), True),
    StructField("file_status", StringType(), False),
    StructField("validation_error", StringType(), True),
])


candidate_total_file_check_df = spark.createDataFrame(
    candidate_total_file_check_rows,
    schema=candidate_total_file_check_schema
)


display(
    candidate_total_file_check_df.select(
        "election_year",
        "source_file_name",
        "source_format",
        "configured_extraction_strategy",
        "file_size_bytes",
        "file_status",
        "validation_error"
    ).orderBy("election_year")
)


unavailable_file_count = (
    candidate_total_file_check_df
    .filter(F.col("file_status") != "AVAILABLE")
    .count()
)


print(
    "Candidate-total files:",
    candidate_total_file_check_df.count()
)
print("Unavailable files:", unavailable_file_count)


if unavailable_file_count > 0:
    raise ValueError(
        f"{unavailable_file_count} candidate-total file(s) are unavailable."
    )

In [0]:
# pypdf extracts the text-based 2016 PDF.
# PyMuPDF renders the scanned 2021 PDF pages into images.
# pytesseract converts those images into raw OCR text.
# Pillow handles the page images in memory.
%pip install pypdf==6.10.0 PyMuPDF==1.26.7 pytesseract==0.3.13 Pillow==11.3.0

In [0]:
# Import and report each extraction dependency.
import pypdf
import fitz
import pytesseract
from PIL import Image

from pypdf import PdfReader


print("pypdf version:", pypdf.__version__)
print("PyMuPDF version:", fitz.VersionBind)
print(
    "Tesseract version:",
    pytesseract.get_tesseract_version()
)
print("Candidate-total extraction dependencies: READY")

In [0]:
# The schema preserves both candidate rows and summary rows exactly as extracted.
# Candidate totals, NOTA and other values remain strings in Bronze.
CANDIDATE_TOTALS_BRONZE_SCHEMA = StructType([
    # Source lineage
    StructField("project_name", StringType(), False),
    StructField("source_file_name", StringType(), False),
    StructField("source_file_path", StringType(), False),
    StructField("source_type", StringType(), False),

    # Stable election metadata
    StructField("constituency_name", StringType(), False),
    StructField("constituency_no", StringType(), False),
    StructField("election_year", IntegerType(), False),

    # Location inside the source
    StructField("page_no", IntegerType(), True),
    StructField("row_no", IntegerType(), True),

    # Raw record classification from the source
    StructField("record_type_raw", StringType(), True),
    StructField("serial_no_raw", StringType(), True),

    # Candidate-level values when present
    StructField("candidate_name_raw", StringType(), True),
    StructField("party_name_raw", StringType(), True),
    StructField("votes_polled_raw", StringType(), True),

    # Full preserved source content
    StructField("raw_row_text", StringType(), True),
    StructField("raw_fields_json", StringType(), True),

    # Extraction diagnostics
    StructField(
        "extraction_metadata_json",
        StringType(),
        True
    ),

    # Parsing outcome
    StructField("parse_status", StringType(), False),
    StructField("parse_error", StringType(), True),
    StructField("extraction_method", StringType(), False),

    # Run lineage
    StructField("batch_id", StringType(), False),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("extraction_timestamp", TimestampType(), True),
])


print(
    "Candidate-total Bronze columns:",
    len(CANDIDATE_TOTALS_BRONZE_SCHEMA.fields)
)
print("Target table:", BRONZE_TABLE)

In [0]:
def clean_raw_text(raw_text):
    """
    Apply technical cleanup only.

    Bronze keeps source wording and does not standardize candidate names,
    parties, vote values or record types.
    """
    if raw_text is None:
        return ""

    return " ".join(
        raw_text.replace("\x00", "").split()
    ).strip()


def build_candidate_total_record(
    source,
    page_no,
    row_no,
    raw_row_text,
    extraction_method,
    parse_status="SUCCESS",
    parse_error=None,
    record_type_raw=None,
    serial_no_raw=None,
    candidate_name_raw=None,
    party_name_raw=None,
    votes_polled_raw=None,
    raw_fields=None,
    extraction_metadata=None,
):
    """
    Build one record that conforms to the stable Bronze schema.

    Centralizing this structure keeps PDF, OCR and CSV extraction consistent.
    """
    return {
        "project_name": source["project_name"],
        "source_file_name": source["source_file_name"],
        "source_file_path": source["source_file_path"],
        "source_type": source["source_type"],
        "constituency_name": source["constituency_name"],
        "constituency_no": source["constituency_no"],
        "election_year": source["election_year"],
        "page_no": page_no,
        "row_no": row_no,
        "record_type_raw": record_type_raw,
        "serial_no_raw": serial_no_raw,
        "candidate_name_raw": candidate_name_raw,
        "party_name_raw": party_name_raw,
        "votes_polled_raw": votes_polled_raw,
        "raw_row_text": raw_row_text,
        "raw_fields_json": (
            json.dumps(raw_fields, ensure_ascii=False)
            if raw_fields is not None
            else None
        ),
        "extraction_metadata_json": (
            json.dumps(
                extraction_metadata,
                ensure_ascii=False
            )
            if extraction_metadata is not None
            else None
        ),
        "parse_status": parse_status,
        "parse_error": parse_error,
        "extraction_method": extraction_method,
        "batch_id": source["batch_id"],
        "ingestion_timestamp": None,
        "extraction_timestamp": datetime.now(
            timezone.utc
        ),
    }


print("Candidate-total record helpers: READY")

In [0]:
def extract_candidate_totals_pdf_text(source):
    """
    Extract raw text lines from a text-based candidate-total PDF.

    This is Bronze extraction:
    - source lines are preserved;
    - candidate names and vote totals are not normalized;
    - unsuccessful pages still produce failure records.
    """
    extracted_records = []

    try:
        pdf_reader = PdfReader(source["source_file_path"])
        total_pages = len(pdf_reader.pages)

        for page_index, pdf_page in enumerate(
            pdf_reader.pages,
            start=1
        ):
            try:
                page_text = pdf_page.extract_text(
                    extraction_mode="layout"
                ) or ""

                extracted_lines = [
                    clean_raw_text(line)
                    for line in page_text.splitlines()
                    if clean_raw_text(line)
                ]

                # Preserve an empty page as a failure record.
                if not extracted_lines:
                    extracted_records.append(
                        build_candidate_total_record(
                            source=source,
                            page_no=page_index,
                            row_no=None,
                            raw_row_text=None,
                            extraction_method="pypdf_layout_text",
                            parse_status="FAILED",
                            parse_error=(
                                "No text extracted from PDF page."
                            ),
                            record_type_raw="PAGE_EXTRACTION_FAILURE",
                            extraction_metadata={
                                "parser": "pypdf",
                                "pdf_page": page_index,
                                "pdf_page_count": total_pages,
                                "extracted_line_count": 0,
                            },
                        )
                    )

                    continue

                # Store every extracted line independently.
                for row_index, extracted_line in enumerate(
                    extracted_lines,
                    start=1
                ):
                    extracted_records.append(
                        build_candidate_total_record(
                            source=source,
                            page_no=page_index,
                            row_no=row_index,
                            raw_row_text=extracted_line,
                            extraction_method="pypdf_layout_text",
                            record_type_raw="RAW_TEXT_LINE",
                            raw_fields={
                                "line_1": extracted_line
                            },
                            extraction_metadata={
                                "parser": "pypdf",
                                "extraction_mode": "layout",
                                "pdf_page": page_index,
                                "pdf_page_count": total_pages,
                                "extracted_line_count": len(
                                    extracted_lines
                                ),
                            },
                        )
                    )

            except Exception as page_error:
                # Retain page-level extraction failures.
                extracted_records.append(
                    build_candidate_total_record(
                        source=source,
                        page_no=page_index,
                        row_no=None,
                        raw_row_text=None,
                        extraction_method="pypdf_layout_text",
                        parse_status="FAILED",
                        parse_error=str(page_error),
                        record_type_raw="PAGE_EXTRACTION_FAILURE",
                        extraction_metadata={
                            "parser": "pypdf",
                            "pdf_page": page_index,
                        },
                    )
                )

    except Exception as file_error:
        # Retain a file-level failure when the PDF cannot be opened.
        extracted_records.append(
            build_candidate_total_record(
                source=source,
                page_no=None,
                row_no=None,
                raw_row_text=None,
                extraction_method="pypdf_layout_text",
                parse_status="FAILED",
                parse_error=str(file_error),
                record_type_raw="FILE_EXTRACTION_FAILURE",
                extraction_metadata={
                    "parser": "pypdf"
                },
            )
        )

    return extracted_records


print("Native candidate-total PDF extractor: READY")

In [0]:
# BytesIO converts rendered PDF pages into in-memory images.
# This avoids writing temporary image files to storage.
from io import BytesIO


def extract_candidate_totals_pdf_ocr(source):
    """
    Render each scanned PDF page and extract raw text with Tesseract.

    OCR output is preserved as raw lines. Candidate parsing and
    standardization remain deferred to Silver.
    """
    extracted_records = []
    pdf_document = None

    try:
        pdf_document = fitz.open(
            source["source_file_path"]
        )

        total_pages = pdf_document.page_count

        for page_index in range(total_pages):
            page_no = page_index + 1

            try:
                pdf_page = pdf_document.load_page(
                    page_index
                )

                # Render at approximately 300 DPI to improve OCR quality.
                zoom = 300 / 72
                render_matrix = fitz.Matrix(zoom, zoom)

                page_pixmap = pdf_page.get_pixmap(
                    matrix=render_matrix,
                    alpha=False
                )

                page_image = Image.open(
                    BytesIO(page_pixmap.tobytes("png"))
                )

                # PSM 6 treats the page as one structured text block.
                ocr_text = pytesseract.image_to_string(
                    page_image,
                    config="--psm 6"
                ) or ""

                extracted_lines = [
                    clean_raw_text(line)
                    for line in ocr_text.splitlines()
                    if clean_raw_text(line)
                ]

                if not extracted_lines:
                    extracted_records.append(
                        build_candidate_total_record(
                            source=source,
                            page_no=page_no,
                            row_no=None,
                            raw_row_text=None,
                            extraction_method="pymupdf_tesseract_ocr",
                            parse_status="FAILED",
                            parse_error="OCR produced no usable text.",
                            record_type_raw="PAGE_EXTRACTION_FAILURE",
                            extraction_metadata={
                                "renderer": "pymupdf",
                                "ocr_engine": "tesseract",
                                "ocr_page_segmentation_mode": 6,
                                "render_dpi": 300,
                                "pdf_page": page_no,
                                "pdf_page_count": total_pages,
                                "extracted_line_count": 0,
                            },
                        )
                    )

                    continue

                for row_index, extracted_line in enumerate(
                    extracted_lines,
                    start=1
                ):
                    extracted_records.append(
                        build_candidate_total_record(
                            source=source,
                            page_no=page_no,
                            row_no=row_index,
                            raw_row_text=extracted_line,
                            extraction_method="pymupdf_tesseract_ocr",
                            record_type_raw="RAW_OCR_LINE",
                            raw_fields={
                                "line_1": extracted_line
                            },
                            extraction_metadata={
                                "renderer": "pymupdf",
                                "ocr_engine": "tesseract",
                                "ocr_page_segmentation_mode": 6,
                                "render_dpi": 300,
                                "pdf_page": page_no,
                                "pdf_page_count": total_pages,
                                "extracted_line_count": len(
                                    extracted_lines
                                ),
                            },
                        )
                    )

            except Exception as page_error:
                extracted_records.append(
                    build_candidate_total_record(
                        source=source,
                        page_no=page_no,
                        row_no=None,
                        raw_row_text=None,
                        extraction_method="pymupdf_tesseract_ocr",
                        parse_status="FAILED",
                        parse_error=str(page_error),
                        record_type_raw="PAGE_EXTRACTION_FAILURE",
                        extraction_metadata={
                            "renderer": "pymupdf",
                            "ocr_engine": "tesseract",
                            "pdf_page": page_no,
                        },
                    )
                )

    except Exception as file_error:
        extracted_records.append(
            build_candidate_total_record(
                source=source,
                page_no=None,
                row_no=None,
                raw_row_text=None,
                extraction_method="pymupdf_tesseract_ocr",
                parse_status="FAILED",
                parse_error=str(file_error),
                record_type_raw="FILE_EXTRACTION_FAILURE",
                extraction_metadata={
                    "renderer": "pymupdf",
                    "ocr_engine": "tesseract",
                },
            )
        )

    finally:
        if pdf_document is not None:
            pdf_document.close()

    return extracted_records


print("OCR candidate-total PDF extractor: READY")

In [0]:
# Python CSV parser preserves quoted fields more reliably than splitting on commas.
import csv


def extract_candidate_totals_csv(source):
    """
    Preserve every physical CSV row, including metadata and summary rows.

    Bronze does not discard preambles or classify election totals as business
    measures. It stores the original row and column positions for later parsing.
    """
    extracted_records = []

    try:
        with open(
            source["source_file_path"],
            "r",
            encoding="utf-8-sig",
            errors="replace",
            newline=""
        ) as csv_file:
            csv_reader = csv.reader(csv_file)

            for row_index, source_row in enumerate(
                csv_reader,
                start=1
            ):
                # Preserve the full physical row in a readable form.
                raw_row_text = " | ".join(
                    value for value in source_row
                )

                # Keep every source column by its position.
                raw_fields = {
                    f"col_{column_index}": value
                    for column_index, value in enumerate(
                        source_row,
                        start=1
                    )
                }

                extracted_records.append(
                    build_candidate_total_record(
                        source=source,
                        page_no=None,
                        row_no=row_index,
                        raw_row_text=raw_row_text,
                        extraction_method="python_csv_reader",
                        record_type_raw="RAW_CSV_ROW",
                        raw_fields=raw_fields,
                        extraction_metadata={
                            "parser": "python_csv",
                            "csv_row_number": row_index,
                            "detected_column_count": len(
                                source_row
                            ),
                            "encoding": "utf-8-sig",
                        },
                    )
                )

        # An empty CSV must still be visible as a failed Bronze record.
        if not extracted_records:
            extracted_records.append(
                build_candidate_total_record(
                    source=source,
                    page_no=None,
                    row_no=None,
                    raw_row_text=None,
                    extraction_method="python_csv_reader",
                    parse_status="FAILED",
                    parse_error="CSV source contained no rows.",
                    record_type_raw="FILE_EXTRACTION_FAILURE",
                    extraction_metadata={
                        "parser": "python_csv",
                        "encoding": "utf-8-sig",
                    },
                )
            )

    except Exception as file_error:
        extracted_records.append(
            build_candidate_total_record(
                source=source,
                page_no=None,
                row_no=None,
                raw_row_text=None,
                extraction_method="python_csv_reader",
                parse_status="FAILED",
                parse_error=str(file_error),
                record_type_raw="FILE_EXTRACTION_FAILURE",
                extraction_metadata={
                    "parser": "python_csv"
                },
            )
        )

    return extracted_records


print("Candidate-total CSV extractor: READY")

In [0]:
# Route each source to the extraction method declared in the manifest.
all_candidate_total_records = []


for source in sorted(
    candidate_total_source_rows,
    key=lambda row: row["election_year"]
):
    extraction_strategy = source[
        "configured_extraction_strategy"
    ]

    print(
        f"Extracting {source['election_year']}: "
        f"{source['source_file_name']}"
    )
    print("Configured strategy:", extraction_strategy)

    if extraction_strategy == "pdf_text":
        source_records = (
            extract_candidate_totals_pdf_text(source)
        )

    elif extraction_strategy == "ocr_required":
        source_records = (
            extract_candidate_totals_pdf_ocr(source)
        )

    elif extraction_strategy == (
        "csv_with_preamble_and_summary_rows"
    ):
        source_records = (
            extract_candidate_totals_csv(source)
        )

    else:
        # Unknown strategies produce a traceable failure record.
        source_records = [
            build_candidate_total_record(
                source=source,
                page_no=None,
                row_no=None,
                raw_row_text=None,
                extraction_method="unsupported_strategy",
                parse_status="FAILED",
                parse_error=(
                    "Unsupported extraction strategy: "
                    f"{extraction_strategy}"
                ),
                record_type_raw="FILE_EXTRACTION_FAILURE",
                extraction_metadata={
                    "configured_extraction_strategy": (
                        extraction_strategy
                    )
                },
            )
        ]

    print("Records extracted:", len(source_records))

    all_candidate_total_records.extend(
        source_records
    )


# Convert the combined records into the stable Bronze dataframe.
candidate_totals_bronze_df = spark.createDataFrame(
    all_candidate_total_records,
    schema=CANDIDATE_TOTALS_BRONZE_SCHEMA
).withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)


# Show bounded extraction coverage rather than every raw line.
display(
    candidate_totals_bronze_df
    .groupBy(
        "election_year",
        "source_file_name",
        "record_type_raw",
        "parse_status",
        "extraction_method"
    )
    .count()
    .orderBy(
        "election_year",
        "record_type_raw",
        "parse_status"
    )
)


print(
    "Total candidate-total Bronze records:",
    candidate_totals_bronze_df.count()
)

In [0]:
from pyspark.sql.window import Window

In [0]:
# Ensure the Bronze schema exists.
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}
""")


# Create an empty Delta table with the stable schema on the first run.
if not spark.catalog.tableExists(BRONZE_TABLE):
    (
        spark.createDataFrame(
            [],
            schema=CANDIDATE_TOTALS_BRONZE_SCHEMA
        )
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(BRONZE_TABLE)
    )


# Remove this batch before inserting it again.
# This makes rerunning this write cell idempotent for the same BATCH_ID.
spark.sql(f"""
DELETE FROM {BRONZE_TABLE}
WHERE batch_id = '{BATCH_ID}'
""")


# Append the validated raw candidate-total records.
(
    candidate_totals_bronze_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(BRONZE_TABLE)
)


print(
    "Candidate-total Bronze rows written:",
    candidate_totals_bronze_df.count()
)
print("Batch ID:", BATCH_ID)
print("Bronze table:", BRONZE_TABLE)

In [0]:
# Read this batch back from Delta.
written_candidate_totals_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("batch_id") == BATCH_ID)
)


# Show the stored coverage by source and extraction method.
display(
    written_candidate_totals_df
    .groupBy(
        "election_year",
        "source_file_name",
        "record_type_raw",
        "parse_status",
        "extraction_method"
    )
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("page_no").alias(
            "page_count"
        )
    )
    .orderBy(
        "election_year",
        "record_type_raw",
        "parse_status"
    )
)


expected_record_count = (
    candidate_totals_bronze_df.count()
)

written_record_count = (
    written_candidate_totals_df.count()
)

distinct_year_count = (
    written_candidate_totals_df
    .select("election_year")
    .distinct()
    .count()
)

failed_record_count = (
    written_candidate_totals_df
    .filter(F.col("parse_status") == "FAILED")
    .count()
)


print("Expected records:", expected_record_count)
print("Written records:", written_record_count)
print("Election years represented:", distinct_year_count)
print("Failed records retained:", failed_record_count)


if written_record_count != expected_record_count:
    raise ValueError(
        "Written row count does not match the validated dataframe."
    )

if distinct_year_count != 3:
    raise ValueError(
        "The written batch does not contain all three election years."
    )


print("Candidate-total Bronze validation: PASS")